In [ ]:
import pandas as pd
import sys
import importlib
from pathlib import Path

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append("../src")
import eda_utils as eda
import visualization as visual
import data_splitting as split
import preprocessing as prep
import constants as const
import knn as knn

importlib.reload(eda)
importlib.reload(visual)
importlib.reload(split)
importlib.reload(prep)
importlib.reload(const)
importlib.reload(knn)

In [ ]:
dataset = pd.read_csv("../data/pf_suvs.csv")

<h1 style="
    background-color: #d0ebff; 
    color: #1a1a1a; 
    display: inline-block; 
    padding: 10px 18px; 
    border-radius: 10px;
    font-size: 32px;
">
Exploratory Data Analysis
</h1>

In [ ]:
# Columns visualization with their types
feature_types = eda.feature_types_summary(dataset)
display(feature_types.style.hide(axis="index"))

In [ ]:
# Shape of the dataset
print(f"Cantidad de filas: {dataset.shape[0]}")
print(f"Cantidad de columnas: {dataset.shape[1]}")

In [ ]:
TARGET = "Precio"

X = dataset.drop(columns=[TARGET])
y = dataset[TARGET]


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Target Analysis
</h3>

In [ ]:
target_summary = eda.explore_target(y, currency = X["Moneda"])
display(target_summary.style.hide(axis="index"))

#### Análisis de la tabla

Analizamos la distribución de la variable objetivo (`Precio`) y la segmentamos por moneda de publicación. En la tabla se observa que no hay valores faltantes ni precios iguales a cero en la variable objetivo. Además, las publicaciones en pesos presentan valores mucho más altos que las publicaciones en dólares, evidenciando una diferencia de órdenes de magnitud entre ambas monedas. Por este motivo, resulta necesario unificar la unidad monetaria antes del modelado.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Analysis
</h3>

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Numeric Feature Statistics
</div>

Esta tabla permite anticipar algunas decisiones importantes para el preprocesamiento. En particular, una de las variables parece comportarse como un posible identificador, ya que sus valores van desde 0 hasta la cantidad de muestras del dataset. Además, en `Año` y `Puertas` aparecen valores máximos claramente inconsistentes (`436694` y `60252`), que no son posibles en el contexto del problema. Esto sugiere la presencia de errores u outliers que deberán revisarse antes del modelado.

In [ ]:
display(X.describe().T.style.format("{:,.2f}"))

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Duplicate Rows Analysis
</div>

Verificamos la existencia de filas completamente duplicadas en el conjunto de datos para identificar posibles publicaciones repetidas o errores de recolección

In [ ]:
eda.duplicate_rows_summary(dataset).style.hide(axis="index")

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Missing Values Analysis
</div>

La mayoría de las variables presentan una cantidad de valores faltantes muy reducida. La principal excepción es `Con cámara de retroceso`, con un 74,30% de datos ausentes. Por este motivo, esta variable requerirá un tratamiento particular durante el preprocesamiento.

In [ ]:
eda.missing_values_summary(dataset).style.hide(axis="index")

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Feature Cardinality Analysis
</div>

Calculamos la cantidad y el porcentaje de valores únicos de cada variable para identificar posibles columnas constantes, columnas indices, variables de texto y atributos categóricos de baja o alta cardinalidad.

`Unnamed: 0` tiene un valor único por fila, por lo que funciona como un identificador. En cambio, `Título` y `Descripción` muestran una alta diversidad de valores, algo esperable en columnas de texto libre escritas por los vendedores.

In [ ]:
eda.unique_values_summary(dataset).style.hide(axis="index")

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Semantic Repetitions Analysis
</div>

Analizamos las variables categóricas en busca de valores semánticamente equivalentes escritos de distintas maneras (por ejemplo, diferencias de mayúsculas, errores tipográficos o variantes de escritura). El objetivo es detectar categorías que representan el mismo concepto y deberían unificarse durante el preprocesamiento

In [ ]:
categorical_columns = [
    "Marca",
    "Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

eda.find_semantic_repetitions(
    dataset,
    columns=categorical_columns,
    similarity_threshold=0.80,
).style.hide(axis="index")


#### Análisis de la tabla

La tabla evidencia numerosas inconsistencias de representación, especialmente en las variables `Color`, `Marca` y `Modelo`. Se observan diferencias de capitalización (`Blanco`, `BLANCA`, `blanca`), errores tipográficos (`Renault` y `Rrenault`, `Jetour` y `Jetur`) y variantes de nomenclatura (`2008` y `208`, `X5` y `X55`). Estas inconsistencias fragmentan artificialmente las categorías y aumentan innecesariamente su cardinalidad, por lo que resulta conveniente normalizarlas antes del modelado

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Initial Graphics Analysis
</h3>

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Categorical Variables
</div>

Para que el gráfico sea más legible, no incluimos `Título`, `Descripción` ni `Versión`. Como se observó en el análisis de cardinalidad, estas columnas tienen una gran cantidad de valores únicos porque contienen texto libre o descripciones específicas de cada publicación. Si se graficaran directamente, la visualización quedaría demasiado cargada y sería difícil interpretar patrones generales.

In [ ]:
visual.plot_currency_counts(dataset)
visual.plot_categorical_counts(X, ignored_columns = ["Título", "Descripción", "Versión"], top_n = 10, n_cols = 2)

#### Análisis gráfico 1
Se observa que una mayor cantidad de publicaciones se encuentran expresadas en pesos que en dólares. Dado que el precio objetivo está representado en ambas monedas, resulta necesario unificar la unidad monetaria antes del modelado para evitar diferencias artificiales de escala.

#### Análisis gráfico 2
Los gráficos muestran que las variables categóricas no se distribuyen de la misma manera. Algunas están muy concentradas en pocas categorías, como `Tipo de combustible`, donde predominan los vehículos nafteros, y `Transmisión`, donde la mayoría de las publicaciones corresponden a cajas automáticas. También se observa que `Con cámara de retroceso` tiene una gran cantidad de valores faltantes, por lo que será necesario tratar esta variable con cuidado en el preprocesamiento.

Otras variables, como `Marca`, `Modelo`, `Motor`, `Color` y `Tipo de vendedor`, presentan más variedad de categorías. Aunque con este análisis no podemos asegurar que todas tengan relación con el target, sí parece razonable conservarlas y evaluarlas durante el modelado, ya que podrían reflejar diferencias entre tipos de vehículos, características del auto o formas de publicación.


<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Numerical Variables and Target Distribution
</div>

Para mejorar la visualización de las distribuciones, se filtraron algunos valores fuera de rango o extremadamente altos. Sin este recorte, la escala de los gráficos quedaba dominada por pocos casos extremos y no permitía observar con claridad la forma general de las curvas.

Este comportamiento reafirma la existencia de posibles outliers, ya que unos pocos valores fueron suficientes para modificar notablemente la escala de las visualizaciones.

In [ ]:
visual.plot_price_distribution_by_currency(dataset)
visual.plot_raw_numeric_distributions(dataset)

#### Análisis gráfico 1
Las distribuciones de `Precio` muestran una asimetría positiva en ambas monedas: la mayoría de las publicaciones se concentra en valores relativamente bajos, mientras que hay una cola hacia precios más altos. Esto indica que existen algunos vehículos bastante más caros que el resto.

Este patrón sugiere la presencia de posibles valores extremos y muestra que la variable objetivo no sigue una distribución normal. Por eso, será importante revisar más adelante que pasa cuando se le aplique logaritmo al `Precio`, intentando llevarla a una forma normal.

#### Análisis gráfico 2
La distribución de `Año` está concentrada en vehículos relativamente recientes. La mayoría de las publicaciones corresponde a modelos posteriores a 2015, con un pico marcado en los años más nuevos del dataset. Esto sugiere que la base está compuesta principalmente por SUVs modernas.

En el caso de `Kilómetros`, la distribución es asimétrica hacia la derecha: hay muchas publicaciones con bajo kilometraje y una cola que se extiende hacia valores más altos. También se observa una cantidad importante de vehículos con kilometraje nulo o cercano a cero, lo que probablemente corresponde a unidades nuevas o con muy poco uso.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Preliminary Outlier Detection
</div>

In [ ]:
visual.plot_preliminary_outliers(dataset)

#### Análisis del gráfico
Los boxplots permiten identificar de forma preliminar la presencia de valores atípicos y posibles errores de carga en las variables numéricas. En `Precio` se observan algunos valores muy elevados en comparación con la mayor parte de las publicaciones, especialmente al analizar los precios separados por moneda.

Además, `Año` y `Puertas` presentan valores claramente imposibles para el contexto del problema, muy superiores a los rangos esperados, como habíamos anticipado en secciones anteriores

<h1 style="
    background-color: #d0ebff; 
    color: #1a1a1a; 
    display: inline-block; 
    padding: 10px 18px; 
    border-radius: 10px;
    font-size: 32px;
">
Post Preprocessing - Exploratory Data Analysis
</h1>

En esta sección se trabaja con el dataset ya preprocesado. El proceso completo de curado y limpieza de los datos se encuentra documentado en [preprocessing.ipynb](preprocessing.ipynb).

Se cargan tres versiones de los datos:

- `dataset_processed`: dataset completo preprocesado, antes del split.
- `X_train` / `X_val`: splits antes de aplicar one-hot encoding.
- `X_train_encoded` / `X_val_encoded`: splits con one-hot aplicado

In [ ]:
PROCESSED_DATA_PATH = Path("../data/dataset_pp.csv")
SPLITTED_DATA_DIR = Path("../data/processed/splitted")
ONE_HOT_DATA_DIR = Path("../data/processed/one_hot")
TEXT_COLS_TO_DROP = ["Descripción", "Versión", "Título"]

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Loading Data
</div>

In [ ]:
# Dataset before-one hot
dataset_processed = pd.read_csv(PROCESSED_DATA_PATH)

# Pre one-hot split
X_train = pd.read_csv(SPLITTED_DATA_DIR / "X_train.csv")
X_val = pd.read_csv(SPLITTED_DATA_DIR / "X_val.csv")

# Post one-hot split
X_train_encoded = pd.read_csv(ONE_HOT_DATA_DIR / "X_train_encoded.csv")
X_val_encoded = pd.read_csv(ONE_HOT_DATA_DIR / "X_val_encoded.csv")

# Targets
y_train = pd.read_csv(SPLITTED_DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(SPLITTED_DATA_DIR / "y_val.csv").squeeze()

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Dropping Text Columns
</div>

In [ ]:
X_train = prep.drop_irrelevant_columns(X_train.copy(), columns_to_drop=TEXT_COLS_TO_DROP)
X_val = prep.drop_irrelevant_columns(X_val.copy(), columns_to_drop=TEXT_COLS_TO_DROP)

X_train_encoded = prep.drop_irrelevant_columns(X_train_encoded.copy(), columns_to_drop=TEXT_COLS_TO_DROP)
X_val_encoded = prep.drop_irrelevant_columns(X_val_encoded.copy(), columns_to_drop=TEXT_COLS_TO_DROP)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Checking Sizes
</div>

In [ ]:
print(f"Dataset preprocesado: {dataset_processed.shape[0]} filas y {dataset_processed.shape[1]} columnas")
print(f"Train pre-one-hot: {X_train.shape}")
print(f"Validation pre-one-hot: {X_val.shape}")
print(f"Train one-hot: {X_train_encoded.shape}")
print(f"Validation one-hot: {X_val_encoded.shape}")

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Pre-One Hot Analysis
</h3>

In [ ]:
train_pre_onehot = visual.build_plot_dataset(X_train, y_train)

visual.plot_price_distribution(train_pre_onehot)
visual.plot_price_distribution(train_pre_onehot, log_transform=True)

visual.plot_clean_numeric_distributions(train_pre_onehot)

visual.plot_price_vs_numeric(train_pre_onehot, x_col="Año", sample_size=5000)
visual.plot_price_vs_numeric(train_pre_onehot, x_col="Kilómetros", sample_size=5000)

visual.plot_year_kilometers_price_scatter(train_pre_onehot, sample_size=5000)

visual.plot_numeric_correlation_heatmap(
    train_pre_onehot,
    feature_type="numeric",
    title="Correlación de variables numéricas antes del one-hot",
)

visual.plot_median_price_by_category(train_pre_onehot, "Marca", top_n=15, min_count=50)
visual.plot_price_boxplot_by_category(train_pre_onehot, "Marca", top_n=15, min_count=50)

visual.plot_median_price_by_category(train_pre_onehot, "Modelo", top_n=15, min_count=80)
visual.plot_median_price_by_category(train_pre_onehot, "Color", top_n=15, min_count=30)

visual.plot_median_price_by_category(train_pre_onehot, "Transmisión", min_count=30)
visual.plot_median_price_by_category(train_pre_onehot, "Tipo de combustible", min_count=30)
visual.plot_median_price_by_category(train_pre_onehot, "Tipo de vendedor", min_count=30)

visual.plot_category_frequency_after_cleaning(
    train_pre_onehot,
    columns=["Marca", "Modelo", "Color", "Transmisión", "Tipo de combustible", "Tipo de vendedor"],
    top_n=15,
)

visual.plot_median_price_by_year_lines(train_pre_onehot, group_col="Marca", top_n=8, min_count=150)
visual.plot_median_price_by_year_lines(train_pre_onehot, group_col="Modelo", top_n=8, min_count=120)

brand_iqr = visual.plot_iqr_ranking_by_category(
    train_pre_onehot,
    category_col="Marca",
    top_n=15,
    min_count=50,
)

brand_iqr

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Post-preprocessing EDA
</h3>

Luego del preprocesamiento, se vuelve a explorar la distribución de las variables principales. Esto permite evaluar cómo quedaron los datos luego de la limpieza y detectar si todavía existen valores extremos que puedan afectar el entrenamiento de los modelos.

In [ ]:
train_encoded = visual.build_plot_dataset(X_train_encoded, y_train)

visual.plot_numeric_and_binary_correlation_heatmap(
    train_encoded,
    title="Correlación de variables numéricas y binarias",
)

color_cols = [col for col in train_encoded.columns if col.startswith("Color_")]
brand_cols = [col for col in train_encoded.columns if col.startswith("Marca_")]

top_color_cols = (
    train_encoded[color_cols]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

top_brand_cols = (
    train_encoded[brand_cols]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

marca_modelo_corr = visual.plot_one_hot_group_correlation_heatmap(
    train_encoded,
    row_prefix="Marca",
    col_prefix="Modelo",
    top_n_pairs=50,
    top_n_rows=18,
    top_n_cols=18,
    min_frequency=0.005,
    title="Correlación entre Marca y Modelo",
    annotate=True,
)


visual.plot_numeric_correlation_heatmap(
    train_encoded,
    numeric_cols=["Precio"] + top_color_cols,
    add_log_price=False,
    include_target=False,
    include_log_target=False,
    title="Correlación entre Precio y colores",
)


visual.plot_numeric_correlation_heatmap(
    train_encoded,
    numeric_cols=["Precio"] + top_brand_cols,
    add_log_price=False,
    include_target=False,
    include_log_target=False,
    title="Correlación entre Precio y Marca",
)

numeric_corr_table = visual.plot_top_target_correlations(
    train_encoded,
    feature_type="numeric",
    top_n=20,
    title="Variables numéricas más correlacionadas con el target",
)

all_corr_table = visual.plot_top_target_correlations(
    train_encoded,
    feature_type="both",
    top_n=25,
    title="Features más correlacionadas con el target",
)